# Glue Interactive Sessions - Batch Processing (Full Load)

This notebook connects to an **AWS Glue interactive session** (account `331504768406`, region `us-east-1`) and runs the **same steps as the `Processor` class** (`processor.run` / `main._run_batch`) in **batch** mode: reading the full-load Parquet from the landing bucket, quality validation, rejected-record writes and Delta MERGE into the raw layer.

**Prerequisites:**

- Local **Glue PySpark** kernel: `pip install jupyter boto3 aws-glue-sessions` and then `install-glue-kernels`. Open with `jupyter notebook` and select the `Glue PySpark` kernel.
- Valid AWS credentials with Glue and S3 access (user `lake-admin` / group `datalake-admins`).
- Role `role-datalake-analytics` with interactive session permissions (already provisioned via Terraform in `infra/iam.tf`).
- `helpers.zip` published in the workspace bucket (`aws-glue/jobs/flight-radar/src/dependencies/helpers.zip`).

> **Delta Lake**: the `%%configure` cell below is required - the project's `writer.py` imports `delta.tables`. Without it the import fails with `ModuleNotFoundError: No module named 'delta'`.


In [ ]:
# 1) Session configuration (AWS Glue kernel magics)
# The next cell (%%configure) enables Delta Lake. Cell magics
# (%%configure/%%tags) must be the first line of the cell, with no comments.
%glue_version 5.0
%iam_role arn:aws:iam::331504768406:role/role-datalake-analytics
%region us-east-1
%worker_type G.1X
%number_of_workers 2
%idle_timeout 30
%session_id_prefix flight-radar-batch

In [ ]:
%%configure
{
  "--datalake-formats": "delta",
  "--conf": "spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension --conf spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog --conf spark.delta.logStore.class=org.apache.spark.sql.delta.storage.S3SingleDriverLogStore"
}

In [ ]:
%%tags
{"Environment": "production", "Project": "flight-radar-glue", "Mode": "batch"}

In [ ]:
# Project dependencies (helpers.zip already published in the workspace bucket)
%extra_py_files s3://lakehouse-workspace-331504768406/aws-glue/jobs/flight-radar/src/dependencies/helpers.zip

In [ ]:
# Instantiate the SparkSession / GlueContext
# In interactive sessions Spark already exists; getOrCreate returns the same session.
from pyspark.context import SparkContext
from awsglue.context import GlueContext

sc = SparkContext.getOrCreate()
glue_context = GlueContext(sc)
spark = glue_context.spark_session

print('Spark version:', spark.version)

In [ ]:
# Project dependencies loaded via %extra_py_files (helpers.zip)
# The zip contains the `src` package (src.dependencies.*).
import boto3
from src.dependencies.processor import Processor
from src.dependencies.config_models import Config

account_id = boto3.client("sts").get_caller_identity()["Account"]
config = Config.from_s3(f"s3://lakehouse-workspace-{account_id}/aws-glue/jobs/flight-radar/src/dependencies/config/config.json")

for s in config.sources:
    print(f"{s.order:>2}  {s.source:<20} -> {s.target.database}.{s.target.table}")

## Steps equivalent to `Processor.run(..., mode='batch')`

`main._run_batch` executes, per table: (1) read via `reader.read(mode='batch')`, (2) `processor.run(source, target, mode='batch', dataframe=raw_df)` - which internally performs quality validation, rejected-record writes and Delta MERGE. Below each step is executed in a separate cell.

In [ ]:
# Choose the table to test (here: aircraft, which has data in the landing bucket)
processor = Processor(spark)

source = config.get_source('aircraft')
target = source.target
print(source.source, '->', f'{target.database}.{target.table}')

In [ ]:
# Step 1 - Batch read of the full-load Parquet (reader.read with mode='batch')
raw_df = processor._reader.read(source, mode='batch')
print('Rows read:', raw_df.count())
raw_df.printSchema()

In [ ]:
raw_df.show(15, truncate=False)

In [ ]:
# Step 2 - Data quality validation (DataQuality.validate)
valid_df, rejects_df = processor._data_quality.validate(raw_df, target, source)
print('Valid:', valid_df.count())
print('Rejected:', 0 if rejects_df.isEmpty() else rejects_df.count())

In [ ]:
# Step 3 - Write the rejected records (writer.write_rejects)
processor._writer.write_rejects(rejects_df, target)
print('Rejects written to:', target.rejected_location)

In [ ]:
# Step 4 - Delta write with MERGE (writer.write)
# Generates cod_unico, derives event_date, projects the target columns and applies
# WHEN NOT MATCHED AND Op <> 'D' INSERT / WHEN MATCHED AND Op='D' DELETE / UPDATE ALL.
processor._writer.write(valid_df, target, source)
print('Delta MERGE completed for', f'{target.database}.{target.table}')

In [ ]:
# Verification - read the final Delta table (via path, independent of the Catalog)
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, target.location)
print('Columns:', dt.toDF().columns)
print('Rows in table:', dt.toDF().count())
dt.toDF().show(5)

## Run the full pipeline (all tables)

Mirrors `main._run_batch`: processes each table in sequence, continuing even if one fails.

In [ ]:
# Full pipeline in batch - equivalent to main._run_batch
for idx, s in enumerate(config.sources, start=1):
    print(f'[{idx}/{len(config.sources)}] {s.source}')
    try:
        df = processor._reader.read(s, mode='batch')
        processor.run(s, s.target, mode='batch', dataframe=df)
        print('  OK')
    except Exception as exc:
        print('  FAILED:', exc)

In [ ]:
# Session status (shows tags, role, workers, region)
%status

In [ ]:
# Stop the session when done
%stop_session